# Additionnel · RQ-014 contrôlée contre l'API de la BAN

**Règle (dq.regle)** : *RQ-014, adresse de sinistre : format valide, rapprochée du référentiel national (BAN), enregistrement bloqué si échec.* Bloquante, porte S, seuil 100 %.

**Ce que fait ce notebook**
1. Lit `silver.sinistre` (produite par S4) et sépare le **format** (code postal à 5 chiffres, numéro et voie présents) du **rapprochement** avec la BAN.
2. Interroge l'API de géocodage de la BAN (IGN, `data.geopf.fr`) : une requête par voie distincte (code postal + commune + voie).
3. Compare avec le référentiel local `silver.ref_ban` utilisé en S4 (US-26).
4. Contrôle le **numéro** sur un échantillon, et vérifie si le référentiel du TD passe lui-même à la BAN.
5. Dit ce que deviendrait le certificat, **sans rien écrire** (lecture seule par défaut).

**Contraintes respectées** : aucune bibliothèque à installer (urllib, json, difflib… + pandas et `aq.py` déjà présents). S2 et S4 ne sont pas modifiés. `dq.dq_history` n'est modifiée que si `ECRIRE = True`.

**Prérequis** : S4 exécuté (`silver.sinistre`, `silver.ref_ban`) ; accès Internet du conteneur vers `data.geopf.fr`. Sans Internet, le notebook rejoue le cache `reports/ban_cache.json`.

## 1. Paramètres

In [3]:
# -*- coding: utf-8 -*-
import sys, os, re, json, time, math, unicodedata, threading, difflib
import urllib.request, urllib.parse, urllib.error
from concurrent.futures import ThreadPoolExecutor
import sys
sys.path.append("../pipelines")          # notebook dans notebooks/ ; en terminal, le dossier pipelines/ est déjà dans le chemin
from pipelines.aq import *                 # lire, mesurer, BATCH, RACINE, pd (déjà installés : aucune nouvelle bibliothèque)

# ---------- paramètres ----------
URL_BAN     = "https://data.geopf.fr/geocodage/search"   # service de géocodage de l'IGN (successeur de api-adresse.data.gouv.fr)
SEUIL_VOIE  = 0.85     # similarité minimale (0 à 1) entre la voie saisie et la voie renvoyée par la BAN
ECH_NUMERO  = 500      # adresses dont on contrôle aussi le numéro (0 : pas de contrôle du numéro)
ECH_REFERENTIEL = 100  # adresses du référentiel local passées à la BAN (0 : sauter)
NUMERO_OBLIGATOIRE = True   # une adresse sans numéro est non conforme (comme en S4, US-26)
MAX_APPELS  = 20000    # garde-fou : le notebook s'arrête avant d'appeler plus que cela
DEBIT_MAX   = 20       # requêtes par seconde (la BAN en accepte 50 par adresse IP)
THREADS     = 6
SEED        = 42
ECRIRE      = False    # True : écrit la mesure RQ-014 (SILVER, sinistre) dans dq.dq_history du lot réel. Voir section 8 avant de passer à True.

CACHE = f"{RACINE}/reports/ban_cache.json"
os.makedirs(os.path.dirname(CACHE), exist_ok=True)
cache = json.load(open(CACHE, encoding="utf-8")) if os.path.exists(CACHE) else {}
print(f"Lot {BATCH} · API {URL_BAN} · cache : {len(cache)} réponses déjà enregistrées")

Lot L-2026-10-01 · API https://data.geopf.fr/geocodage/search · cache : 0 réponses déjà enregistrées


## 2. Outils : normalisation, appel de l'API, règle de décision

La BAN renvoie, pour une requête, le meilleur résultat avec son **type** : `housenumber` (le numéro existe), `street` (la voie existe, pas le numéro), `locality`, `municipality`.
Une adresse est **reconnue au niveau de la voie** si le premier résultat est une voie ou un numéro, avec le **même code postal**, la **même commune** et une **voie ressemblante** (abréviations développées : R = RUE…).
Elle est reconnue **au niveau du numéro** si, en plus, le résultat est de type `housenumber` avec le même numéro.

In [4]:
ABREV = {"R": "RUE", "AV": "AVENUE", "AVE": "AVENUE", "BD": "BOULEVARD", "BLD": "BOULEVARD", "CHE": "CHEMIN", "IMP": "IMPASSE",
         "PL": "PLACE", "ALL": "ALLEE", "RTE": "ROUTE", "SQ": "SQUARE", "CRS": "COURS", "ST": "SAINT", "STE": "SAINTE"}

def norm(s):
    """Majuscules, sans accents, ponctuation et tirets remplacés par des espaces, abréviations courantes développées."""
    s = unicodedata.normalize("NFKD", "" if s is None or (isinstance(s, float) and math.isnan(s)) else str(s)).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    return " ".join(ABREV.get(m, m) for m in s.split())

# ---------- appel de l'API (urllib : bibliothèque standard) ----------
_verrou, _dernier = threading.Lock(), [0.0]
def _creneau():
    """Répartit les appels dans le temps : au plus DEBIT_MAX requêtes par seconde, tous threads confondus."""
    with _verrou:
        t = time.monotonic()
        _dernier[0] = max(t, _dernier[0] + 1.0 / DEBIT_MAX)
        attente = _dernier[0] - t
    if attente > 0:
        time.sleep(attente)

def reduire(rep):
    """On ne garde du JSON que ce qui sert à la décision : le cache reste petit et relisible."""
    cles = ("type", "housenumber", "street", "name", "postcode", "city", "score")
    return {"features": [{"properties": {k: f["properties"].get(k) for k in cles}} for f in (rep.get("features") or [])[:1]]}

def appeler(q, cp):
    """Premier résultat de la BAN pour la requête q, filtré sur le code postal cp. Réponse mise en cache (clé : cp|q)."""
    cle = f"{cp}|{q}"
    if cle in cache:
        return cache[cle]
    url = URL_BAN + "?" + urllib.parse.urlencode({"q": q, "postcode": cp, "limit": 1})
    derniere = None
    for essai in range(4):
        _creneau()
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "TD-AssurConnect-RQ014/1.0"})
            with urllib.request.urlopen(req, timeout=15) as r:
                cache[cle] = reduire(json.load(r))
            return cache[cle]
        except urllib.error.HTTPError as e:
            derniere = e
            if e.code == 400:                                  # requête refusée (trop courte, caractères) : pas de résultat
                cache[cle] = {"features": []}
                return cache[cle]
            time.sleep(2 ** essai)                             # 429, 5xx : on patiente puis on réessaie
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            derniere = e
            time.sleep(2 ** essai)
    raise RuntimeError(f"API BAN injoignable après 4 essais ({derniere}). Rejouez : les réponses déjà reçues sont dans le cache.")

def sauver_cache():
    json.dump(cache, open(CACHE, "w", encoding="utf-8"), ensure_ascii=False)

# ---------- règle de décision ----------
def evaluer(rep, cp, ville, voie, numero=None):
    """Compare le premier résultat de la BAN à l'adresse saisie."""
    f = (rep or {}).get("features") or []
    if not f:
        return {"cause": "aucun résultat (voie ou couple code postal / commune inconnu)", "voie_ok": False, "numero_ok": False, "voie_ban": None, "type_ban": None}
    p = f[0]["properties"]
    voie_ban = p.get("street") or p.get("name") or ""
    sim = difflib.SequenceMatcher(None, norm(voie), norm(voie_ban)).ratio()
    cp_ok = str(p.get("postcode")) == str(cp)
    ville_ok = norm(p.get("city")) == norm(ville)
    voie_ok = p.get("type") in ("street", "housenumber") and cp_ok and ville_ok and sim >= SEUIL_VOIE
    numero_ok = bool(voie_ok and numero is not None and p.get("type") == "housenumber" and str(p.get("housenumber")) == str(numero))
    cause = "reconnue" if voie_ok else ("code postal différent" if not cp_ok else "commune différente" if not ville_ok else "voie différente")
    return {"cause": cause, "voie_ok": bool(voie_ok), "numero_ok": numero_ok, "voie_ban": voie_ban, "type_ban": p.get("type")}

def wilson(k, n, z=1.96):
    """Intervalle de confiance à 95 % d'une proportion k/n (en %)."""
    if n == 0:
        return (None, None)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(100 * (c - h), 1), round(100 * (c + h), 1))

### Auto-test de la règle de décision (sans réseau)

In [5]:
def _f(type_, street, cp, city, hn=None):
    return {"features": [{"properties": {"type": type_, "street": street, "postcode": cp, "city": city, "housenumber": hn}}]}

assert evaluer(_f("housenumber", "Rue de la Paix", "75002", "Paris", "12"), "75002", "PARIS", "R DE LA PAIX", "12")["numero_ok"]
assert evaluer(_f("street", "Rue de la Paix", "75002", "Paris"), "75002", "PARIS", "RUE DE LA PAIX", "9999")["voie_ok"]
assert not evaluer(_f("street", "Rue de la Paix", "75002", "Paris"), "75002", "PARIS", "RUE DE LA PAIX", "9999")["numero_ok"]
assert evaluer(_f("street", "Rue de la Paix", "75002", "Paris"), "75003", "PARIS", "RUE DE LA PAIX")["cause"] == "code postal différent"
assert evaluer(_f("street", "Rue Molière", "01000", "Bourg-en-Bresse"), "01000", "LYON", "RUE MOLIERE")["cause"] == "commune différente"
assert evaluer(_f("street", "Rue Molière", "01000", "Bourg-en-Bresse"), "01000", "BOURG EN BRESSE", "RUE VICTOR HUGO")["cause"] == "voie différente"
assert evaluer({"features": []}, "01000", "X", "Y")["voie_ok"] is False
assert norm("Saint-Étienne") == "SAINT ETIENNE" and norm("ST ETIENNE") == "SAINT ETIENNE"
print("Règle de décision : 8 cas vérifiés.")

Règle de décision : 8 cas vérifiés.


## 3. L'API répond-elle ?

In [6]:
try:
    t0 = time.time()
    essai = appeler("8 bd du port Amiens", "80000")
    p = essai["features"][0]["properties"] if essai["features"] else {}
    print(f"API joignable en {time.time() - t0:.2f} s · résultat : {p.get('type')} · {p.get('housenumber')} {p.get('street')} {p.get('postcode')} {p.get('city')} · score {p.get('score')}")
    ACCES_API = True
except RuntimeError as e:
    ACCES_API = False
    print("Pas d'accès à l'API depuis ce conteneur :", e)
    print("Le notebook continue avec le cache existant (", len(cache), "réponses). Les adresses absentes du cache ne pourront pas être contrôlées.")
sauver_cache()

API joignable en 0.44 s · résultat : housenumber · 8 Boulevard du Port 80000 Amiens · score 0.5965941258741259


## 4. Les adresses de sinistre en silver : format, puis voies à interroger

Le **format** (volet « format valide » de RQ-014) ne nécessite aucun appel : code postal à 5 chiffres, adresse renseignée, numéro présent.
Seules les **voies distinctes** (code postal + commune + voie) sont envoyées à la BAN : les sinistres d'une même voie partagent la même réponse.

In [7]:
sin = lire("SELECT silver_row_id, sinistre_id, canal_saisie, cp, ville_normalisee AS ville, adresse_normalisee FROM silver.sinistre")
sin["adresse_normalisee"] = sin.adresse_normalisee.fillna("").astype(str).str.strip()
sin["numero"] = sin.adresse_normalisee.str.extract(r"^(\d+)")[0]
sin["voie"] = sin.adresse_normalisee.str.replace(r"^\d+\s+", "", regex=True).str.strip()
sin["cp"] = sin.cp.fillna("").astype(str).str.strip()
sin["ville"] = sin.ville.fillna("").astype(str).str.strip()

sin["format_ok"] = sin.cp.str.fullmatch(r"[0-9]{5}") & (sin.voie != "") & (sin.ville != "") & (sin.numero.notna() if NUMERO_OBLIGATOIRE else True)
sin["cause"] = None
sin.loc[~sin.cp.str.fullmatch(r"[0-9]{5}"), "cause"] = "format : code postal sans 5 chiffres"
sin.loc[sin.cause.isna() & (sin.adresse_normalisee == ""), "cause"] = "format : adresse vide"
sin.loc[sin.cause.isna() & NUMERO_OBLIGATOIRE & sin.numero.isna(), "cause"] = "format : pas de numéro"
sin.loc[sin.cause.isna() & ((sin.voie == "") | (sin.ville == "")), "cause"] = "format : voie ou commune vide"

rues = sin.loc[sin.format_ok, ["cp", "ville", "voie"]].drop_duplicates().reset_index(drop=True)
a_appeler = sum(f"{r.cp}|{r.voie} {r.ville}" not in cache for r in rues.itertuples())
print(f"{len(sin):,} sinistres en silver".replace(",", " "))
print(f"  format non conforme (sans appel) : {int((~sin.format_ok).sum()):,}".replace(",", " "))
print(sin.loc[~sin.format_ok, "cause"].value_counts().to_string())
print(f"{len(rues):,} voies distinctes à contrôler, dont {a_appeler:,} pas encore en cache (≈ {a_appeler / DEBIT_MAX / 60:.1f} min au débit de {DEBIT_MAX}/s)".replace(",", " "))
assert a_appeler <= MAX_APPELS, f"{a_appeler} appels prévus : augmentez MAX_APPELS si c'est voulu."

100 000 sinistres en silver
  format non conforme (sans appel) : 3 947
cause
format : pas de numéro           1982
format : voie ou commune vide    1965
13 831 voies distinctes à contrôler  dont 13 831 pas encore en cache (≈ 11.5 min au débit de 20/s)


## 5. Contrôle des voies auprès de la BAN

In [8]:
def controler_voie(r):
    cp, ville, voie = r
    return evaluer(appeler(f"{voie} {ville}", cp), cp, ville, voie)

t0, resultats = time.time(), []
couples = list(zip(rues.cp, rues.ville, rues.voie))
try:
    with ThreadPoolExecutor(THREADS) as ex:
        for i in range(0, len(couples), 500):                       # par lots de 500 : on voit avancer et on sauve le cache
            resultats += list(ex.map(controler_voie, couples[i:i + 500]))
            sauver_cache()
            print(f"  {min(i + 500, len(couples)):>6} / {len(couples)} voies · {time.time() - t0:.0f} s")
except RuntimeError as e:
    sauver_cache()
    raise
print(f"Terminé en {time.time() - t0:.0f} s · {len(cache)} réponses en cache")

res = pd.concat([rues, pd.DataFrame(resultats)], axis=1).rename(columns={"cause": "cause_ban"})
sin = sin.merge(res, on=["cp", "ville", "voie"], how="left")
sin["conforme"] = sin.format_ok & sin.voie_ok.eq(True)           # une voie jamais interrogée (format non conforme) reste non conforme
sin.loc[sin.format_ok & ~sin.conforme, "cause"] = "BAN : " + sin.cause_ban.fillna("")
sin.loc[sin.conforme, "cause"] = "conforme"

     500 / 13831 voies · 37 s
    1000 / 13831 voies · 67 s
    1500 / 13831 voies · 99 s
    2000 / 13831 voies · 137 s
    2500 / 13831 voies · 169 s
    3000 / 13831 voies · 200 s
    3500 / 13831 voies · 230 s
    4000 / 13831 voies · 266 s
    4500 / 13831 voies · 297 s
    5000 / 13831 voies · 333 s
    5500 / 13831 voies · 375 s
    6000 / 13831 voies · 413 s
    6500 / 13831 voies · 442 s
    7000 / 13831 voies · 477 s
    7500 / 13831 voies · 514 s
    8000 / 13831 voies · 544 s
    8500 / 13831 voies · 576 s
    9000 / 13831 voies · 605 s
    9500 / 13831 voies · 636 s
   10000 / 13831 voies · 676 s
   10500 / 13831 voies · 708 s
   11000 / 13831 voies · 748 s
   11500 / 13831 voies · 784 s
   12000 / 13831 voies · 817 s
   12500 / 13831 voies · 850 s
   13000 / 13831 voies · 884 s
   13500 / 13831 voies · 917 s
   13831 / 13831 voies · 950 s
Terminé en 950 s · 13832 réponses en cache


## 6. Résultat : taux d'erreur RQ-014 (BAN) et comparaison avec le référentiel local de S4

In [9]:
testees, echecs = len(sin), int((~sin.conforme).sum())
taux_ban = 100 * echecs / testees
print(f"RQ-014 · adresses de sinistre non conformes (format + voie à la BAN) : {echecs:,} sur {testees:,} = {taux_ban:.2f} %   (conformité {100 - taux_ban:.3f} %)".replace(",", " "))
print("\nCauses :"); print(sin.cause.value_counts().to_string())

# ---- référentiel local, définition de S4 (US-26.1) : (code postal, commune, numéro, voie) présent dans silver.ref_ban ----
def maj_pd(s):
    return (s.astype("string").str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii").str.upper()
             .str.replace(r"\s+", " ", regex=True).str.strip())
ban = lire("SELECT numero, voie, code_postal AS cp, commune AS ville FROM silver.ref_ban")
ban["numero"], ban["voie"], ban["ville"] = ban.numero.astype(str), maj_pd(ban.voie), maj_pd(ban.ville)
cles_ban = set(zip(ban.cp, ban.ville, ban.numero, ban.voie))
sin["local_ok"] = [(a, b, c, d) in cles_ban for a, b, c, d in zip(sin.cp, sin.ville, sin.numero.astype(str), sin.voie)]
taux_local = 100 * (~sin.local_ok).mean()

print(f"\nRéférentiel local (S4, US-26.1) : {taux_local:.2f} % non reconnues")
print(f"API BAN (voie, après format)    : {taux_ban:.2f} % non conformes")
print("\nLocal contre BAN (lignes) :")
print(pd.crosstab(sin.local_ok.map({True: "local : reconnue", False: "local : non reconnue"}), sin.conforme.map({True: "BAN : conforme", False: "BAN : non conforme"})).to_string())

print("\nPar canal de saisie :")
canal = sin.groupby("canal_saisie").agg(sinistres=("conforme", "size"), non_conformes=("conforme", lambda x: int((~x).sum())))
canal["taux_%"] = (100 * canal.non_conformes / canal.sinistres).round(2)
print(canal.sort_values("non_conformes", ascending=False).to_string())

RQ-014 · adresses de sinistre non conformes (format + voie à la BAN) : 82 484 sur 100 000 = 82.48 %   (conformité 17.516 %)

Causes :
cause
BAN : voie différente                                                  69264
conforme                                                               17516
BAN : aucun résultat (voie ou couple code postal / commune inconnu)     4771
BAN : commune différente                                                4498
format : pas de numéro                                                  1982
format : voie ou commune vide                                           1965
BAN : code postal différent                                                4

Référentiel local (S4, US-26.1) : 11.87 % non reconnues
API BAN (voie, après format)    : 82.48 % non conformes

Local contre BAN (lignes) :
conforme              BAN : conforme  BAN : non conforme
local_ok                                                
local : non reconnue            1050               10821
local : 

**Lecture.** Le référentiel local compare l'adresse **entière** (numéro compris) à 22 856 adresses : c'est un contrôle de numéro, très strict. La BAN, ici, contrôle que la **voie existe dans la commune et le code postal indiqués** : c'est un contrôle de voie. Les deux taux ne mesurent donc pas la même chose. La section 7 mesure le numéro séparément.

## 7. Le numéro : échantillon de sinistres, puis référentiel local lui-même

In [10]:
def controler_numero(r):
    cp, ville, voie, numero = r
    return evaluer(appeler(f"{numero} {voie} {ville}", cp), cp, ville, voie, numero)

num_res = None
if ECH_NUMERO:
    base = sin.loc[sin.conforme, ["cp", "ville", "voie", "numero"]].drop_duplicates()
    ech = base.sample(min(ECH_NUMERO, len(base)), random_state=SEED).reset_index(drop=True)
    a_faire = sum(f"{r.cp}|{r.numero} {r.voie} {r.ville}" not in cache for r in ech.itertuples())
    assert a_faire <= MAX_APPELS
    with ThreadPoolExecutor(THREADS) as ex:
        num_res = pd.DataFrame(list(ex.map(controler_numero, zip(ech.cp, ech.ville, ech.voie, ech.numero))))
    sauver_cache()
    k, n = int(num_res.numero_ok.sum()), len(num_res)
    print(f"Sinistres dont la voie est reconnue : le numéro existe dans la BAN pour {k} adresses sur {n} = {100 * k / n:.1f} %  (IC 95 % : {wilson(k, n)[0]} à {wilson(k, n)[1]} %)")
    print(num_res.type_ban.value_counts(dropna=False).to_string())

ref_res = None
if ECH_REFERENTIEL:
    ref = ban.drop_duplicates().sample(min(ECH_REFERENTIEL, len(ban)), random_state=SEED).reset_index(drop=True)
    with ThreadPoolExecutor(THREADS) as ex:
        ref_res = pd.DataFrame(list(ex.map(controler_numero, zip(ref.cp, ref.ville, ref.voie, ref.numero))))
    sauver_cache()
    k, n = int(ref_res.numero_ok.sum()), len(ref_res)
    kv = int(ref_res.voie_ok.sum())
    print(f"\nRéférentiel local passé à la BAN ({n} adresses tirées) : voie reconnue {kv}/{n} = {100 * kv / n:.1f} %, numéro reconnu {k}/{n} = {100 * k / n:.1f} %")

Sinistres dont la voie est reconnue : le numéro existe dans la BAN pour 84 adresses sur 500 = 16.8 %  (IC 95 % : 13.8 à 20.3 %)
type_ban
street         413
housenumber     87

Référentiel local passé à la BAN (100 adresses tirées) : voie reconnue 24/100 = 24.0 %, numéro reconnu 3/100 = 3.0 %


**Comment lire la section 7.** Si le référentiel local lui-même n'est reconnu au niveau du numéro que dans une faible part des cas, ses numéros sont fictifs (jeu de données du TD) : un échec de numéro à la BAN ne prouve alors pas une erreur de saisie. Dans ce cas, la mesure retenue pour RQ-014 est celle de la **voie** (section 6), et le numéro reste un indicateur d'information.

## 8. Ce que deviendrait le certificat (lecture seule)

Le certificat est GO si toutes les règles **bloquantes** des portes B, S et G sont conformes sur la couche de leur porte. RQ-014 est bloquante, porte S, seuil 100 % : **une seule adresse non conforme suffit à la faire échouer.** On simule ici, sans écrire.

In [11]:
regle = lire("SELECT rule_id, severite, porte, seuil_min FROM dq.regle WHERE rule_id = 'RQ-014'").iloc[0]
DECISION = """
SELECT m.rule_id, m.table_cible, m.taux_conformite, r.seuil_min
FROM dq.v_derniere_mesure m JOIN dq.regle r USING (rule_id)
WHERE m.batch_id = :lot AND r.severite = 'BLOQUANTE' AND r.porte IN ('B', 'S', 'G')
  AND m.couche = ELT(FIELD(r.porte, 'B', 'S', 'G'), 'BRONZE', 'SILVER', 'GOLD') AND m.rule_id <> 'RQ-014'"""
actuel = lire(DECISION, lot=BATCH)
ok_actuel = int((actuel.taux_conformite.astype(float) >= actuel.seuil_min.astype(float)).sum())
taux_conf = 100 - taux_ban
rq14_ok = taux_conf >= float(regle.seuil_min)
print(f"Certificat sans RQ-014 : {ok_actuel}/{len(actuel)} mesures bloquantes conformes")
print(f"Avec RQ-014 ({regle.severite}, porte {regle.porte}, seuil {float(regle.seuil_min):.0f} %) mesurée à {taux_conf:.3f} % : {ok_actuel + int(rq14_ok)}/{len(actuel) + 1} → {'GO' if ok_actuel == len(actuel) and rq14_ok else 'NO_GO'}")
if not rq14_ok:
    print("RQ-014 fait passer le lot en NO_GO tant que la sévérité reste BLOQUANTE à 100 %.")
    print("Décision de gouvernance, à noter au journal : sévérité (Data Steward), seuil (Data Owner), ou traitement des adresses (quarantaine, correction BAN).")

Certificat sans RQ-014 : 36/36 mesures bloquantes conformes
Avec RQ-014 (BLOQUANTE, porte S, seuil 100 %) mesurée à 17.516 % : 36/37 → NO_GO
RQ-014 fait passer le lot en NO_GO tant que la sévérité reste BLOQUANTE à 100 %.
Décision de gouvernance, à noter au journal : sévérité (Data Steward), seuil (Data Owner), ou traitement des adresses (quarantaine, correction BAN).


### Écriture dans `dq.dq_history` (facultative)

`ECRIRE = True` ajoute la mesure **(SILVER, RQ-014, sinistre)** au lot réel, avec `mesurer()` de `aq.py` (rejouable : elle remplace la mesure précédente). Elle porte sur **toutes** les lignes de `silver.sinistre`, au niveau de la voie.
Le certificat existant n'est **pas** modifié : pour qu'il tienne compte de RQ-014, il faut relancer la cellule d'émission de S5 (US-31). Si RQ-014 est sous 100 %, le nouveau certificat sera NO_GO.

In [12]:
if ECRIRE:
    print(mesurer("SILVER", "RQ-014", "sinistre", testees, echecs))
    print("Écrit dans dq.dq_history. Relancer S5 (US-31) pour réémettre le certificat.")
else:
    print("ECRIRE = False : rien n'a été écrit dans dq.dq_history. Le certificat et le lot sont inchangés.")

ECRIRE = False : rien n'a été écrit dans dq.dq_history. Le certificat et le lot sont inchangés.


## 9. Preuves à conserver (reports/)

In [13]:
os.makedirs(f"{RACINE}/reports", exist_ok=True)
resume = pd.DataFrame([{
    "lot": BATCH, "mesure_le": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "api": URL_BAN, "seuil_similarite_voie": SEUIL_VOIE,
    "testees": testees, "non_conformes": echecs, "taux_erreur_ban_pct": round(taux_ban, 3), "taux_erreur_local_s4_pct": round(taux_local, 3),
    "voies_distinctes": len(rues),
    "numero_ok_echantillon_sinistres_pct": None if num_res is None else round(100 * num_res.numero_ok.mean(), 1),
    "numero_ok_referentiel_local_pct": None if ref_res is None else round(100 * ref_res.numero_ok.mean(), 1),
    "ecrit_dans_dq_history": bool(ECRIRE)}])
resume.to_csv(f"{RACINE}/reports/rq014_ban_resume.csv", index=False, encoding="utf-8")
sin.loc[~sin.conforme, ["sinistre_id", "canal_saisie", "cp", "ville", "adresse_normalisee", "cause"]].to_csv(f"{RACINE}/reports/rq014_non_conformes.csv", index=False, encoding="utf-8")
print(resume.T.to_string(header=False))
print(f"\nFichiers : reports/rq014_ban_resume.csv · reports/rq014_non_conformes.csv ({echecs} lignes) · reports/ban_cache.json (réponses de la BAN, rejouables sans Internet)")

lot                                                            L-2026-10-01
mesure_le                                                  2026-11-20 13:38
api                                  https://data.geopf.fr/geocodage/search
seuil_similarite_voie                                                  0.85
testees                                                              100000
non_conformes                                                         82484
taux_erreur_ban_pct                                                  82.484
taux_erreur_local_s4_pct                                             11.871
voies_distinctes                                                      13831
numero_ok_echantillon_sinistres_pct                                    16.8
numero_ok_referentiel_local_pct                                         3.0
ecrit_dans_dq_history                                                 False

Fichiers : reports/rq014_ban_resume.csv · reports/rq014_non_conformes.csv (82484 lignes